# 23 — Data Ranking / 数据排名

**Chapter 23 — File 1 of 1**

## Summary / 摘要

Data ranking converts numerical values into their ordinal positions, fundamental for non-parametric statistical methods. The scipy.stats.rankdata function assigns ranks from 1 to n, with special handling for tied values. Ranking is essential for distribution-free tests (Mann-Whitney U, Spearman correlation) that don't assume normality. The method='average' parameter assigns the mean rank to tied values, preventing rank bias. Rankings reveal relative ordering independent of scale, enabling robust statistical inference when distributional assumptions fail.

数据排名将数值转换为其序数位置，对非参数统计方法至关重要。scipy.stats.rankdata函数分配从1到n的排名，对平局值有特殊处理。排名对于不假设正态性的无分布测试(Mann-Whitney U、Spearman相关性)至关重要。method='average'参数将平均排名分配给平局值，防止排名偏差。排名独立于规模揭示相对顺序，在分布假设失败时实现稳健的统计推断。

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import required libraries
# 导入所需库
import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt

# Set random seed for reproducibility
# 设置随机种子以保证可重复性
np.random.seed(42)

## Step 2 — Generate Sample Data / 生成样本数据

In [ ]:
# Generate random data with some ties
# 生成有一些平局的随机数据
data = np.array([23, 15, 8, 42, 15, 30, 8, 45, 28, 15, 35, 22, 38])

# Display original data
# 显示原始数据
print(f"Original data: {data}")
print(f"Data length: {len(data)}")
print(f"Data range: [{data.min()}, {data.max()}]")

# Identify unique values and ties
# 识别唯一值和平局
unique_vals, counts = np.unique(data, return_counts=True)
ties = unique_vals[counts > 1]

print(f"\nUnique values: {len(unique_vals)}")
print(f"Values with ties: {ties}")
if len(ties) > 0:
    for val in ties:
        print(f"  Value {val}: appears {np.sum(data == val)} times")

## Step 3 — Perform Data Ranking / 执行数据排名

In [ ]:
# Rank data using average method for ties
# 使用平均方法为平局排名数据
ranks_average = rankdata(data, method='average')

# Also demonstrate other ranking methods
# 还演示其他排名方法
ranks_ordinal = rankdata(data, method='ordinal')  # Ordinal ranking (first occurrence gets lower rank)
                                                   # 序数排名（首次出现获得较低排名）
ranks_min = rankdata(data, method='min')  # Minimum rank for ties / 平局的最小排名
ranks_max = rankdata(data, method='max')  # Maximum rank for ties / 平局的最大排名
ranks_dense = rankdata(data, method='dense')  # Dense ranking (no gaps) / 密集排名（无间隙）

# Display ranking results
# 显示排名结果
print(f"\nRanking Methods Comparison:")
print(f"{'Value':>6} | {'Average':>8} | {'Ordinal':>8} | {'Min':>5} | {'Max':>5} | {'Dense':>6}")
print("-" * 50)
for i, val in enumerate(data):
    print(f"{val:6d} | {ranks_average[i]:8.1f} | {ranks_ordinal[i]:8d} | {ranks_min[i]:5d} | {ranks_max[i]:5d} | {ranks_dense[i]:6d}")

## Step 4 — Analyze Ranking Properties / 分析排名属性

In [ ]:
# Ranking statistics (using average method)
# 排名统计（使用平均方法）
print(f"\nRanking Statistics (Average Method):")
print(f"  Number of observations: {len(data)}")
print(f"  Expected rank range: [1, {len(data)}]")
print(f"  Actual rank range: [{ranks_average.min():.1f}, {ranks_average.max():.1f}]")
print(f"  Mean rank: {ranks_average.mean():.2f} (expected: {(len(data)+1)/2:.2f})")
print(f"  Std of ranks: {ranks_average.std():.2f}")

# Verify all ranks are assigned
# 验证所有排名都已分配
print(f"\nRank assignment verification:")
print(f"  All ranks between 1 and n: {(ranks_average.min() >= 1) and (ranks_average.max() <= len(data))}")
print(f"  Ranks sum to n(n+1)/2: {ranks_average.sum()} = {len(data)*(len(data)+1)//2}")

## Step 5 — Visualize Data and Ranks / 可视化数据和排名

In [ ]:
# Create visualization
# 创建可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Original data
# 图1: 原始数据
indices = np.arange(len(data))
colors = plt.cm.viridis(data / data.max())
axes[0, 0].bar(indices, data, color=colors, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Index')
axes[0, 0].set_ylabel('Value')
axes[0, 0].set_title('Original Data')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(data):
    axes[0, 0].text(i, v + 1, str(v), ha='center', fontsize=9)

# Plot 2: Rankings (Average method)
# 图2: 排名（平均方法）
axes[0, 1].bar(indices, ranks_average, color=colors, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Index')
axes[0, 1].set_ylabel('Rank')
axes[0, 1].set_title('Ranks (Average Method for Ties)')
axes[0, 1].set_ylim([0, len(data) + 1])
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Add rank labels
for i, r in enumerate(ranks_average):
    axes[0, 1].text(i, r + 0.2, f'{r:.1f}', ha='center', fontsize=9)

# Plot 3: Sorted data with ranks
# 图3: 排序数据与排名
sorted_indices = np.argsort(data)
sorted_data = data[sorted_indices]
sorted_ranks = ranks_average[sorted_indices]

ax3 = axes[1, 0]
ax3_twin = ax3.twinx()

bars = ax3.bar(indices, sorted_data, alpha=0.6, color='blue', label='Sorted values')
line = ax3_twin.plot(indices, sorted_ranks, 'ro-', linewidth=2, markersize=6, label='Ranks')

ax3.set_xlabel('Position (after sorting)')
ax3.set_ylabel('Value', color='blue')
ax3_twin.set_ylabel('Rank', color='red')
ax3.set_title('Sorted Data with Corresponding Ranks')
ax3.tick_params(axis='y', labelcolor='blue')
ax3_twin.tick_params(axis='y', labelcolor='red')
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Comparison of ranking methods
# 图4: 排名方法的比较
axes[1, 1].plot(indices, ranks_average, 'o-', linewidth=2, markersize=6, label='Average')
axes[1, 1].plot(indices, ranks_min, 's--', linewidth=1.5, markersize=5, label='Min')
axes[1, 1].plot(indices, ranks_max, '^--', linewidth=1.5, markersize=5, label='Max')
axes[1, 1].plot(indices, ranks_dense, 'D--', linewidth=1.5, markersize=4, label='Dense')

axes[1, 1].set_xlabel('Index')
axes[1, 1].set_ylabel('Rank')
axes[1, 1].set_title('Comparison of Ranking Methods')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
## Learning Notes / 学习笔记

- **Statistical Concept**: Ranking converts data to ordinal positions, eliminating scale dependence. The average method for ties assigns the mean rank to all tied values, preserving sum of ranks = n(n+1)/2 and ensuring mean rank = (n+1)/2. This approach is essential for non-parametric tests that rely on rank differences rather than absolute values, making tests robust to outliers and distribution shape.
  
  **统计概念**: 排名将数据转换为序数位置，消除规模依赖性。平均平局方法将平均排名分配给所有平局值，保持排名总和 = n(n+1)/2，并确保平均排名 = (n+1)/2。此方法对于依赖于排名差异而非绝对值的非参数检验至关重要，使检验对异常值和分布形状稳健。

- **ML Application**: Ranking is the foundation for non-parametric statistical tests and algorithms. In ML, rankings enable: (1) robust feature engineering (rank-based transformations preserve monotonic relationships), (2) ordinal regression for categorical outcomes, (3) ranking-based ensemble methods (e.g., rank averaging). Ranking also handles outliers naturally—extreme values don't distort ranks. In production systems, ranking metrics like rank-AUC are more interpretable for ranking tasks and less sensitive to score calibration.
  
  **ML应用**: 排名是非参数统计测试和算法的基础。在ML中，排名启用：(1)稳健的特征工程（基于排名的转换保留单调关系），(2)用于分类结果的序数回归，(3)基于排名的集成方法（例如排名平均）。排名也自然处理异常值——极端值不会扭曲排名。在生产系统中，像rank-AUC这样的排名指标对于排名任务更可解释，对分数校准不太敏感。

➡️ **Next**: `../chapter_24/01_dataset.ipynb`

## Complete Code / 完整代码一览

In [ ]:
import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt

np.random.seed(42)
data = np.array([23, 15, 8, 42, 15, 30, 8, 45, 28, 15, 35, 22, 38])

ranks = rankdata(data, method='average')

print(f"Data: {data}")
print(f"Ranks: {ranks}")
print(f"Mean rank: {ranks.mean():.2f}")
print(f"Expected: {(len(data)+1)/2:.2f}")

# Verify properties
print(f"Sum of ranks: {ranks.sum()} (expected: {len(data)*(len(data)+1)//2})")